In [1]:
import os
import re
from collections import defaultdict
from typing import List, Dict, Tuple, Set
import multiprocessing as mp
from functools import partial
from tqdm import tqdm

def process_file(file_name: str, directory: str, ignore_list: List[str], pattern: re.Pattern) -> Tuple[str, str, str]:
    full_path = os.path.join(directory, file_name)
    if not os.path.isfile(full_path) or (ignore_list and any(ignore_text in file_name for ignore_text in ignore_list)):
        return None
    name, ext = os.path.splitext(file_name)
    modified_name = pattern.sub(r'\1\2', name)
    return (modified_name, ext, full_path)

def analyze_files(directory: str, ignore_list: List[str] = None, num_processes: int = None) -> Dict[str, Tuple[int, List[str], Set[str]]]:
    if num_processes is None:
        num_processes = mp.cpu_count()

    pattern = re.compile(r'(.*?)[0-9]{6}(.*)')
    
    file_list = os.listdir(directory)
    
    with mp.Pool(processes=num_processes) as pool:
        process_func = partial(process_file, directory=directory, ignore_list=ignore_list, pattern=pattern)
        results = list(tqdm(pool.imap(process_func, file_list), total=len(file_list), desc="Analyzing files"))

    file_info = defaultdict(lambda: [0, set(), set()])
    for result in results:
        if result:
            modified_name, ext, full_path = result
            file_info[modified_name][0] += 1
            file_info[modified_name][1].add(ext)
            file_info[modified_name][2].add(full_path)

    return {k: (v[0], list(v[1]), v[2]) for k, v in file_info.items()}

In [2]:
directory = '/data-net/ted/H2/H2_1_1/'

file_analysis = analyze_files(directory, 
                              ignore_list=['flow', 'warped', 'overlay', '.tar', '.mp4', '_frame', 'AM_image'])

Analyzing files: 100%|██████████| 1528320/1528320 [28:37<00:00, 889.77it/s]  


## Get the Averages and Stdev of each Data Type

In [3]:
import os
import numpy as np
from PIL import Image
from PIL import ImageFile
import multiprocessing as mp
from functools import partial
from typing import Dict, List, Tuple, Any, Set
from tqdm import tqdm

ImageFile.LOAD_TRUNCATED_IMAGES = True

def load_and_process_file(file_path: str, file_type: str) -> np.ndarray:
    if file_type.startswith('rgb') or file_type.startswith('semantic') or file_type.startswith('depth'):
        return np.array(Image.open(file_path))
    elif file_type.startswith('attention'):
        with np.load(file_path) as data:
            return data['attention']
    else:
        raise ValueError(f"Unsupported file type: {file_type}")

def compute_average_and_std(file_paths: List[str], file_type: str) -> Tuple[np.ndarray, np.ndarray, int]:
    with mp.Pool() as pool:
        process_func = partial(load_and_process_file, file_type=file_type)
        data_list = list(tqdm(pool.imap(process_func, file_paths), 
                              total=len(file_paths), 
                              desc=f"Processing {file_type}"))
    
    data_array = np.stack(data_list)
    average_data = np.mean(data_array, axis=0)
    std_data = np.std(data_array, axis=0)
    
    if file_type.startswith('attention'):
        average_data = average_data / (np.sum(average_data) + 1e-8)
    elif file_type.startswith('rgb'):
        average_data = average_data.astype(np.uint8)
        std_data = std_data.astype(np.uint8)
    
    return average_data, std_data, len(file_paths)

def compute_averages_and_std(file_info: Dict[str, Tuple[int, List[str], Set[str]]]) -> Dict[str, Tuple[int, np.ndarray, np.ndarray, str]]:
    averages_and_std = {}
    
    for file_type, (count, extensions, paths) in tqdm(file_info.items(), desc="Computing averages and std"):
        if len(paths) == 0:
            continue
        
        file_paths = list(paths)
        extension = extensions[0] if extensions else ''
        
        try:
            average_data, std_data, num_files = compute_average_and_std(file_paths, file_type)
            averages_and_std[file_type] = (num_files, average_data, std_data, extension)
        except ValueError as e:
            print(f"Skipping {file_type}: {str(e)}")
    
    return averages_and_std

In [4]:
# Assuming we have the file_info dictionary from the previous analyze_files function
averages = compute_averages_and_std(file_analysis)

Computing averages and std:   0%|          | 0/46 [3:45:48<?, ?it/s]


KeyboardInterrupt: 